# Evaluation

Small helpers to present the results in the same way in every notebook.

Imports

In [ ]:
import numpy as np
import pandas as pd

### Cluster numbering

PAM numbers clusters arbitrarily, so they are renumbered with cluster 1 = highest mean MMSE. This keeps the figures comparable across k and across configurations.

In [ ]:
def relabel_by(df, labels, var="MMSE", ascending=False):
    """Clusters renumbered 1..k by their mean of `var` (default: cluster 1 = highest mean MMSE).

    PAM's cluster numbers are arbitrary; a fixed ordering makes figures comparable across k.
    """
    means = pd.Series(df[var].to_numpy()).groupby(labels).mean().sort_values(ascending=ascending)
    mapping = {old: new for new, old in enumerate(means.index, start=1)}
    return np.array([mapping[l] for l in labels])

### PSI of a partition

Computes the PSI on the interpretation variables defined in `config.ipynb`.

In [ ]:
def psi_report(df, labels, name):
    data = df.assign(cluster=labels)
    return compute_config_pairwise_separability(
        data, "cluster", ordinal_vars=cfg.PSI_ORDINAL, nominal_vars=cfg.PSI_NOMINAL, config_name=name
    )

### PSI as a table column

Per-variable PSI plus the aggregate, as one column.

In [ ]:
def psi_column(df, labels, name):
    """Per-variable PSI plus the aggregate, as one column named `name`."""
    report = psi_report(df, labels, name)
    values = {v.variable: v.mean_pairwise_effect for v in report.per_variable}
    values["PSI (mean)"] = report.pairwise_separability_index
    return pd.Series(values, name=name)

### Magnitude label

N (negligible), S (small), M (medium) or L (large).

In [ ]:
def effect_label(value):
    if value >= 0.474:
        return "L"
    if value >= 0.33:
        return "M"
    if value >= 0.147:
        return "S"
    return "N"

### PSI table

Writes each value with its magnitude label, for example `0.718 (L)`.

In [ ]:
def format_psi_table(table):
    """Values with their N/S/M/L magnitude label, e.g. '0.718 (L)'."""
    return table.apply(lambda col: col.map(lambda v: f"{v:.3f} ({effect_label(v)})"))

### Medoids

The medoid patient of each cluster with their clinical values.

In [ ]:
def medoid_table(df, labels, medoids):
    """One row per cluster: its medoid patient and that patient's clinical values."""
    rows = []
    for m in medoids:
        c = labels[m]
        rows.append({
            "cluster": c,
            "n": int((labels == c).sum()),
            "medoid": df.loc[m, "patient_id"],
            "sex": df.loc[m, "M/F"],
            **{v: df.loc[m, v] for v in ["Age", "MMSE", "CDR", "nWBV", "eTIV", "Educ", "SES"]},
        })
    return pd.DataFrame(rows).sort_values("cluster").reset_index(drop=True)